In [97]:
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

In [98]:
q1_gates = ["H", "X", "I", "SX", "T", "Tdg"]
q2_gates = ["CX", "SWAP", "Display"]
q3_gates = ["CCX", "CSWAP"]

In [99]:
def show_results(qc:QuantumCircuit, player_1:str, player_2:str):
    aer_sim = AerSimulator()
    with Session(backend=aer_sim) as session:
        sampler = Sampler(mode=session)
        result = sampler.run([qc], shots=10000).result()

    outputs = result[0].data.meas
    counts = outputs.get_counts()

    display(plot_histogram(counts))

    try:
        A = counts["000"] // 1000
    except KeyError:
        A = 0
    try:
        B = counts["111"] // 1000
    except KeyError:
        B = 0
    print(f"{player_1}: {A} points\n{player_2}: {B} points")
    if(A > B):
        print(f"{player_1} wins!")
    elif(A < B):
        print(f"{player_2} wins!")
    else:
        print("It's a tie!")

In [100]:
def make_move(qc:QuantumCircuit, gate_cost:int):
    cost = -1
    print(f"Remaining Gate points: {gate_cost}")
    while(cost not in range(1,4)):
        print("Enter Gate cost between 1-3.")
        cost = int(input("Gate cost for move: ").strip())
    if cost == 1:
        valid_gate = False
        while(not valid_gate):
            gate = input(f"Pick one of {q1_gates}: ").strip()
            if gate in q1_gates:
                valid_gate = True
        qubit_valid = False
        while(not qubit_valid):
            qubit = -1
            qubit = int(input("Qubit: ").strip())
            if qubit in range(3):
                qubit_valid = True
        gate_cost -= 1
        if gate == "H":
            qc.h(qubit)
            return qc, gate_cost
        elif gate == "X":
            qc.x(qubit)
            return qc, gate_cost
        elif gate == "I":
            return qc, gate_cost
        elif gate == "SX":
            qc.sx(qubit)
            return qc, gate_cost
        elif gate == "T":
            qc.t(qubit)
            return qc, gate_cost
        else:
            qc.tdg(qubit)
            return qc, gate_cost
    elif cost == 2:
        valid_gate = False
        while(not valid_gate):
            gate = input(f"Pick one of {q2_gates}: ").strip()
            if gate in q2_gates:
                valid_gate = True
        if gate == "Display":
            display(qc.draw('mpl'))
            return qc
        qubits_valid = False
        while(not qubits_valid):
            control, target = -1, -1
            control, target = map(int, input("Enter 2 valid qubit positions: ").strip().split())
            if (control in range(3)) and (target in range(3)):
                qubits_valid = True
        gate_cost -= 2
        if gate == "CX":
            qc.cx(control, target)
            return qc, gate_cost
        else:
            qc.swap(control, target)
            return qc, gate_cost
    else:
        valid_gate = False
        while(not valid_gate):
            gate = input(f"Pick one of {q3_gates}: ").strip()
            if gate in q3_gates:
                valid_gate = True
        qubits_valid = False
        while(not qubits_valid):
            q1, q2, q3 = -1, -1, -1
            q1, q2, q3 = map(int, input("Enter 3 valid qubit positions: ").strip().split())
            if (q1 in range(3)) and (q2 in range(3)) and (q3 in range(3)):
                qubits_valid = True
        gate_cost -= 3
        if gate == "CCX":
            qc.ccx(q1, q2, q3)
            return qc, gate_cost
        else:
            qc.cswap(q1, q2, q3)
            return qc, gate_cost

In [101]:
def request_end(requests:int, player_1:str, player_2:str):
    responses = ["Y", "n"]
    valid_reponse = False
    while(not valid_reponse):
        end_1 = input(f"Do you want to end game, {player_1}?(Y/n): ").strip()
        if end_1 in responses:
            valid_reponse = True
    
    if end_1 == "Y":
        requests -= 1
        valid_reponse = False
        while(not valid_reponse):
            end_2 = input(f"Do you want to end game, {player_2}?(Y/n): ").strip()
            if end_2 in responses:
                valid_reponse = True
        if end_2 == "Y":
            return -2
        else:
            return requests
    else:
        return requests

In [ ]:
requests_1, requests_2 = 1, 1
gate_cost_1, gate_cost_2 = 36, 36

print("Welcome to Quantum Tic-Tac-Toe!!")
print("Rules:")
print("1. The game is played on a 3-qubit quantum circuit initialized by |+> state on each qubit.")
print(f"2. Each player can spend a total of {gate_cost_1} gate points apply gates to the quantum circuit and request their opponent {requests_1} time(s) to end the game.")
print("3. Players will take turns and choose one of the given gates to achieve their target state, until\n   either both players want to end game or one of the players runs out of gates and requests.")
print("4. Target state of player 0 is |000> and that of player 1 is |111>. Maximum attainable points is 10.")
print("Let's Play!!!\n")

Player_1 = input("Enter your name, Player 0: ").strip()
Player_2 = input("Enter your name, Player 1: ").strip()

board = QuantumCircuit(3)
board.h(range(3))

player_1_forfeit, player_2_forfeit = False, False

while(True):
    if(not gate_cost_1):
        print(f"{Player_1} is out of gates.")
        board.measure_all()
        display(board.draw("mpl"))
        show_results(qc=board, player_1=Player_1, player_2=Player_2)
        break
    else:
        if(requests_1 in [0,1]):
            print(f"\n--------------------\n{Player_1}, your move\n")
            board, gate_cost_1 = make_move(board, gate_cost_1)
            requests_1 = request_end(requests_1, Player_1, Player_2)
        else:
            if player_1_forfeit == False:
                print(f"{Player_1} ends his game. {Player_2} has to win within 6 Gate points.")
                player_1_forfeit = True
                gate_cost_2 = 6

    if(requests_1 == -2):
        print("Game ends.")
        board.measure_all()
        display(board.draw("mpl"))
        show_results(qc=board, player_1=Player_1, player_2=Player_2)
        break
    
    if(not gate_cost_2):
        print(f"{Player_2} is out of gates.")
        board.measure_all()
        display(board.draw("mpl"))
        show_results(qc=board, player_1=Player_1, player_2=Player_2)
        break
    else:
        if(requests_2 in [0,1]):
            print(f"\n--------------------\n{Player_2}, your move\n")
            board, gate_cost_2 = make_move(board, gate_cost_2)
            requests_2 = request_end(requests_2, Player_2, Player_1)
        else:
            if player_2_forfeit == False:
                print(f"{Player_2} ends his game. {Player_1} has to win within 6 Gate points.")
                player_2_forfeit = True
                gate_cost_1 = 6

    if(requests_2 == -2):
        print("Game ends.")
        board.measure_all()
        display(board.draw("mpl"))
        show_results(qc=board, player_1=Player_1, player_2=Player_2)
        break